In [1]:
import os
import pandas as pd

In [11]:
file_path = "AggregatedGenerationPerType/2019_01_AggregatedGenerationPerType_16.1.B_C_r3.csv"

df = pd.read_csv(file_path, sep='\t')

unique_values = df["AreaDisplayName"].unique()

print("Unique AreaDisplayName values:")
print(unique_values)

Unique AreaDisplayName values:
['Austria (AT)' 'Bosnia and Herz. (BA)' 'Belgium (BE)' 'Bulgaria (BG)'
 'Switzerland (CH)' 'Cyprus (CY)' 'Czech Republic (CZ)' 'Germany (DE)'
 'DE(50Hertz)' 'DE(Amprion)' 'DE-LU' 'DE(TenneT GER)' 'DE(TransnetBW)'
 'Denmark (DK)' 'DK' 'DK1' 'DK2' 'Estonia (EE)' 'Spain (ES)'
 'Finland (FI)' 'France (FR)' 'GB' 'United Kingdom (UK)' 'Greece (GR)'
 'Croatia (HR)' 'Hungary (HU)' 'Ireland (IE)' 'IE(SEM)' 'Italy (IT)'
 'IT-Centre-North' 'IT-Centre-South' 'IT-North' 'IT-Rossano' 'IT-Sardinia'
 'IT-Sicily' 'IT-South' 'Lithuania (LT)' 'Luxembourg (LU)' 'Latvia (LV)'
 'Montenegro (ME)' 'North Macedonia (MK)' 'NIE' 'Netherlands (NL)'
 'Norway (NO)' 'NO1' 'NO2' 'NO3' 'NO4' 'NO5' 'Poland (PL)' 'Portugal (PT)'
 'Romania (RO)' 'Serbia (RS)' 'Sweden (SE)' 'SE1' 'SE2' 'SE3' 'SE4'
 'Slovenia (SI)' 'Slovakia (SK)']


In [ ]:
input_folder = "AggregatedGenerationPerType"
output_folder = "filtered-AggregatedGenerationPerType"

os.makedirs(output_folder, exist_ok=True)

for file_name in os.listdir(input_folder):
    if file_name.endswith(".csv"):
        input_path = os.path.join(input_folder, file_name)
        
        # Read CSV (adjust separator if needed)
        df = pd.read_csv(input_path, sep='\t')
        
        # Filter rows
        df_filtered = df[
            df["AreaDisplayName"].isin(["DE-AT-LU", "DE-LU"])
        ]
        
        # Create new filename (first 7 characters)
        new_name = file_name[:7] + ".csv"
        output_path = os.path.join(output_folder, new_name)
        
        # Handle possible duplicate names
        counter = 1
        while os.path.exists(output_path):
            new_name = f"{file_name[:7]}_{counter}.csv"
            output_path = os.path.join(output_folder, new_name)
            counter += 1
        
        # Save file
        df_filtered.to_csv(output_path, index=False)
        
        print(f"Saved: {output_path}")

In [13]:
input_folder = "filtered-AggregatedGenerationPerType"
output_file = "combined_filtered-AggregatedGenerationPerType.csv"

all_dfs = []

# Read all CSV files
for file_name in os.listdir(input_folder):
    if file_name.endswith(".csv"):
        file_path = os.path.join(input_folder, file_name)
        
        df = pd.read_csv(file_path)
        
        # Ensure DateTime column is parsed correctly
        df["DateTime(UTC)"] = pd.to_datetime(df["DateTime(UTC)"], errors="coerce")
        
        all_dfs.append(df)

# Combine all dataframes
combined_df = pd.concat(all_dfs, ignore_index=True)

# Sort by datetime (oldest to newest)
combined_df = combined_df.sort_values(by="DateTime(UTC)")

# Save as tab-separated file
combined_df.to_csv(output_file, sep="\t", index=False)

print(f"Combined file saved as: {output_file}")

Combined file saved as: combined_filtered-AggregatedGenerationPerType.csv


In [27]:
# Load the CSV file
file_path = "combined_filtered-AggregatedGenerationPerType.csv"
df = pd.read_csv(file_path, sep='\t')

In [28]:
# Loop through each column and print unique values
col ='AreaTypeCode'
print(f"\nColumn: {col}")
unique_vals = df[col].unique()
print(f"Number of unique values: {len(unique_vals)}")
print(unique_vals)


Column: AreaTypeCode
Number of unique values: 1
['BZN']


In [3]:
df = pd.read_csv("combined_filtered-AggregatedGenerationPerType.csv", sep="\t")

# Drop unnecessary columns
df = df.drop(columns=[
    "AreaCode",
    "AreaTypeCode",
    "AreaMapCode",
    "ResolutionCode"  #all are 15 minutes
])

df["ActualConsumption[MW]"] = pd.to_numeric(df["ActualConsumption[MW]"], errors="coerce").fillna(0)

# (optional) save back
df.to_csv("combined_filtered-AggregatedGenerationPerType_clean.csv", sep="\t", index=False)

print("Columns dropped successfully.")
print("Remaining columns:", df.columns.tolist())

Columns dropped successfully.
Remaining columns: ['DateTime(UTC)', 'AreaDisplayName', 'ProductionType', 'ActualGenerationOutput[MW]', 'ActualConsumption[MW]', 'UpdateTime(UTC)']


In [35]:
df = pd.read_csv("combined_filtered-AggregatedGenerationPerType_clean.csv", sep="\t")

print(df.isnull().sum())

DateTime(UTC)                 0
AreaDisplayName               0
ProductionType                0
ActualGenerationOutput[MW]    0
ActualConsumption[MW]         0
UpdateTime(UTC)               0
dtype: int64


In [7]:
df = pd.read_csv("combined_filtered-AggregatedGenerationPerType_clean.csv", sep="\t")
df['ProductionType'].unique()

array(['Nuclear', 'Other renewable', 'Biomass', 'Solar',
       'Hydro Run-of-river and poundage', 'Fossil Hard coal',
       'Fossil Gas', 'Hydro Pumped Storage', 'Geothermal', 'Other',
       'Wind Offshore', 'Fossil Oil', 'Fossil Brown coal/Lignite',
       'Fossil Coal-derived gas', 'Wind Onshore', 'Hydro Water Reservoir',
       'Waste'], dtype=object)

In [4]:
df = pd.read_csv("combined_filtered-AggregatedGenerationPerType_clean.csv", sep="\t")

# Drop further columns to make it smaller
df = df.drop(columns=[
    "AreaDisplayName",
    "UpdateTime(UTC)"
])

# (optional) save back
df.to_csv("final_generation_minimized.csv", sep="\t", index=False)

print("Remaining columns:", df.columns.tolist())


Remaining columns: ['DateTime(UTC)', 'ProductionType', 'ActualGenerationOutput[MW]', 'ActualConsumption[MW]']


In [5]:
# Load data
file_path = "final_generation_minimized.csv"
df = pd.read_csv(file_path, sep="\t")

# Ensure datetime format (optional but recommended)
df["DateTime(UTC)"] = pd.to_datetime(df["DateTime(UTC)"])

# Aggregate
df_agg = (
    df.groupby("DateTime(UTC)", as_index=False)
      .agg({
          "ActualGenerationOutput[MW]": "sum",
          "ActualConsumption[MW]": "sum"
      })
)

# Save back to the same file (overwrite)
df_agg.to_csv(file_path, index=False)
